# 07a · Iteraciones de hiperparámetros (sección 5.2)

Dos barridos, ambos medidos **solo en validación** (julio–septiembre 2014, backtest diario de 92 orígenes); la prueba no se toca:

1. **Ventana L** ∈ {48, 168, 336} h para DLinear y PatchTST.
2. **Meses de historia** ∈ {3, 12, 30} para seasonal naive, LightGBM, DLinear y PatchTST. La historia termina siempre el 2014-06-30, así que 3 meses = desde el 2014-04-01 y 30 meses = todo el entrenamiento.

Cada iteración registra configuración (L, H, meses), pérdida de entrenamiento y validación por epoch/paso, MAE y MASE de validación, parámetros, tiempo de entrenamiento (por epoch y total), tiempo de inferencia sobre los 92 orígenes de validación y memoria pico de GPU. Las otras iteraciones (normalización, tamaño, capas, dropout, lr) están en los notebooks 03, 04 y 06a.

In [ ]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")

import pandas as pd
import torch
import matplotlib.pyplot as plt
from neuralforecast.models import PatchTST
from src import data as D, metrics as M, backtest as B, train as T
from src.dataset import WindowDataset
from src.models import baselines as BL
from src.models.linear import DLinear
from src.models.nf import fit_nf, nf_forecast_fn

torch.set_float32_matmul_precision("high")
device = "cuda" if torch.cuda.is_available() else "cpu"
panel = D.build_panel()
scale = M.mase_scale(panel)
scaler = D.SeriesScaler().fit(panel)
z = scaler.transform(panel)
H = 24
RUNS = D.ROOT / "outputs" / "runs"

# Mejores configuraciones elegidas en los notebooks 03, 04 y 06a
res = {m: json.loads((B.RESULTS_DIR / f"{m}.json").read_text()) for m in ["LightGBM", "DLinear", "PatchTST"]}
lgb_params = res["LightGBM"]["config"]
dlinear_revin = "revin" in res["DLinear"]["config"]
patch_len = 24 if res["PatchTST"]["config"].endswith("p24") else 16
print("LightGBM:", lgb_params, "| DLinear RevIN:", dlinear_revin, "| PatchTST patch_len:", patch_len)

def val_backtest(fn, name):
    fc = B.backtest(fn, panel, name=name, start=D.VAL_START, end=D.VAL_END, verbose=False)
    r = M.evaluate(fc, panel, scale)["resumen"]
    return r["MAE"], r["MASE"], fc.attrs["inference_s"]

def history_start(months):
    return D.TRAIN_END + pd.Timedelta(hours=1) - pd.DateOffset(months=months)

def make_patchtst(L):
    return lambda **kw: PatchTST(h=H, input_size=L, patch_len=patch_len, stride=patch_len // 2, encoder_layers=3,
                                 n_heads=16, hidden_size=128, linear_hidden_size=256, dropout=0.2, revin=True,
                                 learning_rate=5e-4, batch_size=32, windows_batch_size=1024,
                                 scaler_type="identity", random_seed=42, **kw)

def run_dlinear(L, months, name):
    tr = WindowDataset(z, L=L, H=H, split="train", history_start=history_start(months))
    va = WindowDataset(z, L=L, H=H, split="val", origin_hour=0)
    r = T.fit(DLinear(L, H, revin=dlinear_revin), tr, va, scaler, scale, name=name, epochs=15, patience=3,
              lr=1e-3, batch_size=1024, device=device, verbose=False,
              config={"familia": "DLinear", "meses_historia": months})
    mae, mase, inf_s = val_backtest(B.torch_forecast_fn(r.model, scaler, L=L, H=H, device=device), name)
    info = {**r.info, "val_MAE_backtest": mae, "val_MASE_backtest": mase, "inference_s_val92": inf_s}
    (RUNS / name / "info.json").write_text(json.dumps(info, indent=2, default=str))
    return info, r.history

def run_patchtst(L, months, name):
    zz = z[["unique_id", "ds", "y"]]
    nf, info = fit_nf(make_patchtst(L), zz, val_size=2208, max_steps=1500, val_check_steps=100, patience=5,
                      history_start=history_start(months) - pd.Timedelta(hours=L))
    mae, mase, inf_s = val_backtest(nf_forecast_fn(nf, scaler, L, "PatchTST"), name)
    hist = info["train_traj"].merge(info["valid_traj"], on="step", how="left")
    out = {k: v for k, v in info.items() if not k.endswith("_traj")}
    out.update({"nombre": name, "familia": "PatchTST", "L": L, "H": H, "meses_historia": months,
                "val_MAE_backtest": mae, "val_MASE_backtest": mase, "inference_s_val92": inf_s,
                "train_s_total": info["fit_s"], "train_s_por_paso": info["fit_s"] / max(info["pasos"], 1)})
    (RUNS / name).mkdir(parents=True, exist_ok=True)
    hist.to_csv(RUNS / name / "history.csv", index=False)
    (RUNS / name / "info.json").write_text(json.dumps(out, indent=2, default=str))
    del nf; torch.cuda.empty_cache()
    return out, hist

## 1. Barrido de la ventana L (30 meses de historia)

In [ ]:
filas_L, curvas = [], {}
for L in [48, 168, 336]:
    info, hist = run_dlinear(L, 30, f"DLinear_L{L}")
    filas_L.append(info); curvas[f"DLinear_L{L}"] = hist
    print(f"DLinear  L={L:3d}  MASE val {info['val_MASE_backtest']:.4f}")
for L in [48, 168, 336]:
    info, hist = run_patchtst(L, 30, f"PatchTST_L{L}")
    filas_L.append(info); curvas[f"PatchTST_L{L}"] = hist
    print(f"PatchTST L={L:3d}  MASE val {info['val_MASE_backtest']:.4f}  ({info['fit_s']:.0f}s)")

cols = ["familia", "L", "H", "meses_historia", "params_total", "val_MAE_backtest", "val_MASE_backtest",
        "train_s_total", "inference_s_val92", "gpu_mem_pico_MB"]
tab_L = pd.DataFrame(filas_L)[cols]
tab_L.to_csv(RUNS / "barrido_L.csv", index=False)
tab_L

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for name, h in curvas.items():
    if name.startswith("DLinear"):
        line, = axes[0].plot(h["epoch"], h["train_loss"], ls="--", alpha=.7)
        axes[0].plot(h["epoch"], h["val_loss"], color=line.get_color(), label=name)
    else:
        tr = h.set_index("step")["train_loss"].rolling(50, min_periods=1).mean()
        line, = axes[1].plot(tr.index, tr.values, ls="--", alpha=.6)
        va = h.dropna(subset=["val_loss"])
        axes[1].plot(va["step"], va["val_loss"], color=line.get_color(), marker="o", ms=3, label=name)
axes[0].set_title("DLinear: MSE (-- train, — val)"); axes[0].set_xlabel("epoch")
axes[1].set_title("PatchTST: MAE (-- train suavizada, — val)"); axes[1].set_xlabel("paso")
for a in axes: a.grid(alpha=.3); a.legend(fontsize=8)
fig.tight_layout(); fig.savefig(D.ROOT / "outputs" / "figs" / "curvas_barrido_L.png", dpi=150)

## 2. Barrido de meses de historia (L = 168)

In [ ]:
filas_hist = []
for months in [3, 12, 30]:
    hs = history_start(months)
    # Seasonal naive: no se entrena, la historia no cambia nada (línea de referencia)
    mae, mase, inf_s = val_backtest(BL.seasonal_naive_fn(24), "SN24")
    filas_hist.append({"modelo": "SeasonalNaive24", "meses": months, "val_MASE": mase, "train_s": 0.0})
    # LightGBM
    fn, info = BL.fit_lightgbm(panel.loc[panel["ds"] <= D.TRAIN_END], params=lgb_params,
                               history_start=hs - pd.Timedelta(hours=168))
    mae, mase, inf_s = val_backtest(fn, f"LightGBM_{months}m")
    filas_hist.append({"modelo": "LightGBM", "meses": months, "val_MASE": mase, "train_s": info["fit_s"]})
    # DLinear
    info, _ = run_dlinear(168, months, f"DLinear_hist{months}m")
    filas_hist.append({"modelo": "DLinear", "meses": months, "val_MASE": info["val_MASE_backtest"],
                       "train_s": info["train_s_total"]})
    # PatchTST
    info, _ = run_patchtst(168, months, f"PatchTST_hist{months}m")
    filas_hist.append({"modelo": "PatchTST", "meses": months, "val_MASE": info["val_MASE_backtest"],
                       "train_s": info["train_s_total"]})
    print(f"{months:2d} meses listo")

tab_hist = pd.DataFrame(filas_hist)
tab_hist.to_csv(RUNS / "barrido_historia.csv", index=False)
tab_hist.pivot(index="meses", columns="modelo", values="val_MASE")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for m, g in tab_hist.groupby("modelo"):
    ax.plot(g["meses"], g["val_MASE"], marker="o", label=m)
ax.set_xlabel("Meses de historia de entrenamiento"); ax.set_ylabel("MASE (validación)")
ax.set_xticks([3, 12, 30]); ax.grid(alpha=.3); ax.legend()
ax.set_title("MASE contra meses de historia")
fig.tight_layout(); fig.savefig(D.ROOT / "outputs" / "figs" / "mase_vs_historia_A.png", dpi=150)

Cisco agrega a `outputs/runs/barrido_historia.csv` las filas de su mejor recurrente y la línea de Chronos para la gráfica final de la sección 6.